In [0]:
"""
HOME CREDIT | BRONZE -> SILVER | AZURE DATABRICKS

A fully sectioned PySpark job with explanations before each operation.
Unity Catalog: retaildatabricks2026.practical_scenarios
Source: destination/bronze/home_credit/application_train/
Prerequisite: Databricks secret azure-storage / insurance-account-key.

WARNING: Silver and quarantine are full-snapshot OVERWRITES.
Validation and audit logs are APPENDED on every run.
This file is syntax-checked, not executed against your Databricks workspace.
"""


# =====================================================================
# SECTION 01: IMPORTS
# Load Spark SQL functions, window utilities, and run identifiers.
# =====================================================================

import uuid
from datetime import datetime, timezone
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window


# =====================================================================
# SECTION 02: UNITY CATALOG AND AZURE STORAGE SETTINGS
# Keep storage, catalog, and output identifiers in one location.
# =====================================================================

CATALOG = 'retaildatabricks2026'
SCHEMA = 'practical_scenarios'
ACCOUNT = 'insurancedataset2026'
CONTAINER = 'destination'
BRONZE = f'wasbs://{CONTAINER}@{ACCOUNT}.blob.core.windows.net/bronze/home_credit/application_train/'
BASE = f'`{CATALOG}`.`{SCHEMA}`'
RUN_ID = str(uuid.uuid4())
NOW = datetime.now(timezone.utc).replace(tzinfo=None)
checks, audits = [], []



# =====================================================================
# SECTION 03: AUDIT LOG HELPER
# Collect counts for each pipeline stage.
# =====================================================================

def audit(stage, incoming, outgoing, rejected=0):
    audits.append((RUN_ID, stage, int(incoming), int(outgoing), int(rejected), NOW))
    print(f'{stage}: input={incoming}, output={outgoing}, rejected={rejected}')



# =====================================================================
# SECTION 04: VALIDATION AND REJECTION HELPER
# Count failed records and attach rejection reasons without dropping rows.
# =====================================================================

def check(df, name, invalid, total, description):
    invalid = F.coalesce(invalid, F.lit(False))
    failures = df.filter(invalid).count()
    checks.append((RUN_ID, name, 'PASS' if failures == 0 else 'FAIL', int(total), int(failures), description, NOW))
    return df.withColumn('_reasons', F.when(invalid, F.array_union('_reasons', F.array(F.lit(name)))).otherwise(F.col('_reasons')))



# =====================================================================
# SECTION 05: MAIN PIPELINE ENTRY POINT
# Execute all transformations and write Delta outputs.
# =====================================================================

def main():

    # =====================================================================
    # SECTION 06: INITIALIZE SPARK AND CONNECT TO STORAGE
    # Read the storage key from a Databricks secret; never hardcode it.
    # =====================================================================

    spark = SparkSession.builder.getOrCreate()
    spark.conf.set('spark.sql.session.timeZone', 'UTC')
    from pyspark.dbutils import DBUtils
    key = DBUtils(spark).secrets.get(scope='azure-storage', key='insurance-account-key')
    spark.conf.set(f'fs.azure.account.key.{ACCOUNT}.blob.core.windows.net', key)
    spark.sql(f'USE CATALOG `{CATALOG}`')
    spark.sql(f'USE SCHEMA `{SCHEMA}`')

    # =====================================================================
    # SECTION 07: READ BRONZE PARQUET
    # Read all available Bronze columns and count source rows.
    # =====================================================================

    df = spark.read.parquet(BRONZE)
    total = df.count()
    if not total:
        raise ValueError('Bronze is empty')
    audit('BRONZE_READ', total, total)


    # =====================================================================
    # SECTION 08: STANDARDIZE COLUMN NAMES
    # Trim names and convert them to uppercase underscore format.
    # =====================================================================

    names = [c.strip().upper().replace(' ', '_').replace('-', '_') for c in df.columns]
    if len(names) != len(set(names)):
        raise ValueError('Duplicate standardized column names')
    df = df.toDF(*names)

    # =====================================================================
    # SECTION 09: PRESERVE BRONZE LINEAGE
    # Rename technical columns so source metadata survives Silver processing.
    # =====================================================================

    for src, dst in {'_RUN_ID':'BRONZE_RUN_ID', '_SOURCE_FILE':'BRONZE_SOURCE_FILE',
                     '_INGESTION_TIMESTAMP':'BRONZE_INGESTION_TIMESTAMP',
                     '_SOURCE_CONTAINER':'BRONZE_SOURCE_CONTAINER', '_FILE_HASH':'BRONZE_FILE_HASH'}.items():
        if src in df.columns:
            df = df.withColumnRenamed(src, dst)

    # =====================================================================
    # SECTION 10: VALIDATE REQUIRED COLUMNS
    # Fail fast when required business fields are absent.
    # =====================================================================

    required = ['SK_ID_CURR','TARGET','NAME_CONTRACT_TYPE','CODE_GENDER','FLAG_OWN_CAR',
                'FLAG_OWN_REALTY','CNT_CHILDREN','AMT_INCOME_TOTAL','AMT_CREDIT',
                'AMT_ANNUITY','AMT_GOODS_PRICE','DAYS_BIRTH','DAYS_EMPLOYED',
                'EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
    missing = sorted(set(required) - set(df.columns))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    # =====================================================================
    # SECTION 11: INITIALIZE REJECTION TRACKING
    # Create an array to accumulate multiple quality failures per record.
    # =====================================================================

    df = df.withColumn('_reasons', F.expr('cast(array() as array<string>)'))

    # =====================================================================
    # SECTION 12: CLEAN STRING COLUMNS
    # Trim whitespace and convert empty strings to null.
    # =====================================================================

    for field in df.schema.fields:
        if isinstance(field.dataType, T.StringType) and not field.name.startswith('BRONZE_'):
            c = field.name
            df = df.withColumn(c, F.when(F.trim(F.col(c)) == '', None).otherwise(F.trim(F.col(c))))

    # =====================================================================
    # SECTION 13: DEFINE NUMERIC CONVERSION RULES
    # Declare integer and floating-point source fields.
    # =====================================================================

    integer_cols = ['SK_ID_CURR','TARGET','CNT_CHILDREN','DAYS_BIRTH','DAYS_EMPLOYED',
                    'CNT_FAM_MEMBERS','REGION_RATING_CLIENT','REGION_RATING_CLIENT_W_CITY',
                    'HOUR_APPR_PROCESS_START']
    float_cols = ['AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY','AMT_GOODS_PRICE',
                  'EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']

    # =====================================================================
    # SECTION 14: CAST INTEGER COLUMNS
    # Use safe casts and flag values that cannot be converted.
    # =====================================================================

    for c in integer_cols:
        if c in df.columns:
            v = F.expr(f'try_cast(`{c}` as int)')
            df = check(df, f'INVALID_INTEGER_{c}', F.col(c).isNotNull() & v.isNull(), total, 'Integer conversion')
            df = df.withColumn(c, v)

    # =====================================================================
    # SECTION 15: CAST DECIMAL-LIKE COLUMNS
    # Convert amount and score fields to double and flag invalid values.
    # =====================================================================

    for c in float_cols:
        v = F.expr(f'try_cast(`{c}` as double)')
        df = check(df, f'INVALID_DOUBLE_{c}', F.col(c).isNotNull() & (v.isNull() | F.isnan(v)), total, 'Double conversion')
        df = df.withColumn(c, F.when(F.isnan(v), None).otherwise(v))

    # =====================================================================
    # SECTION 16: NORMALIZE CATEGORICAL VALUES
    # Uppercase gender and ownership indicators.
    # =====================================================================

    for c in ['CODE_GENDER','FLAG_OWN_CAR','FLAG_OWN_REALTY']:
        df = df.withColumn(c, F.upper(c))

    # =====================================================================
    # SECTION 17: HANDLE UNKNOWN GENDER
    # Convert XNA to null while retaining a source-quality flag.
    # =====================================================================

    df = df.withColumn('GENDER_UNKNOWN_FLAG', F.when(F.col('CODE_GENDER') == 'XNA', 1).otherwise(0))
    df = df.withColumn('CODE_GENDER', F.when(F.col('CODE_GENDER') == 'XNA', None).otherwise(F.col('CODE_GENDER')))

    # =====================================================================
    # SECTION 18: HANDLE EMPLOYMENT SENTINEL
    # Convert the known 365243 special value to null and retain a flag.
    # =====================================================================

    df = df.withColumn('EMPLOYMENT_SENTINEL_FLAG', F.when(F.col('DAYS_EMPLOYED') == 365243, 1).otherwise(0))
    df = df.withColumn('DAYS_EMPLOYED', F.when(F.col('DAYS_EMPLOYED') == 365243, None).otherwise(F.col('DAYS_EMPLOYED')))

    # =====================================================================
    # SECTION 19: VALIDATE PRIMARY KEY
    # SK_ID_CURR must not be null.
    # =====================================================================

    df = check(df, 'NULL_PRIMARY_KEY', F.col('SK_ID_CURR').isNull(), total, 'SK_ID_CURR required')

    # =====================================================================
    # SECTION 20: VALIDATE TARGET LABEL
    # TARGET must be either 0 or 1.
    # =====================================================================

    df = check(df, 'INVALID_TARGET', F.col('TARGET').isNull() | ~F.col('TARGET').isin(0,1), total, 'TARGET must be 0/1')

    # =====================================================================
    # SECTION 21: VALIDATE POSITIVE FINANCIAL AMOUNTS
    # Income and credit must be greater than zero.
    # =====================================================================

    for c in ['AMT_INCOME_TOTAL','AMT_CREDIT']:
        df = check(df, f'INVALID_{c}', F.col(c).isNull() | (F.col(c) <= 0), total, 'Positive amount required')

    # =====================================================================
    # SECTION 22: VALIDATE NONNEGATIVE AMOUNTS AND COUNTS
    # Reject negative annuity, goods price, or child count.
    # =====================================================================

    for c in ['AMT_ANNUITY','AMT_GOODS_PRICE','CNT_CHILDREN']:
        df = check(df, f'NEGATIVE_{c}', F.col(c) < 0, total, 'Nonnegative required')

    # =====================================================================
    # SECTION 23: DERIVE AND VALIDATE AGE
    # Convert DAYS_BIRTH to years and enforce the configured age range.
    # =====================================================================

    df = df.withColumn('AGE_YEARS', F.round(-F.col('DAYS_BIRTH') / 365.25, 2))
    df = check(df, 'INVALID_AGE', F.col('DAYS_BIRTH').isNull() | (F.col('DAYS_BIRTH') >= 0) |
               (F.col('AGE_YEARS') < 18) | (F.col('AGE_YEARS') > 100), total, 'Age 18-100')

    # =====================================================================
    # SECTION 24: VALIDATE EMPLOYMENT DAYS
    # Reject unexpected positive employment days after sentinel cleanup.
    # =====================================================================

    df = check(df, 'INVALID_EMPLOYMENT', F.col('DAYS_EMPLOYED') > 0, total, 'Unexpected positive days')

    # =====================================================================
    # SECTION 25: DERIVE EMPLOYMENT YEARS
    # Convert negative day offsets into years of employment.
    # =====================================================================

    df = df.withColumn('EMPLOYMENT_YEARS', F.round(-F.col('DAYS_EMPLOYED') / 365.25, 2))

    # =====================================================================
    # SECTION 26: VALIDATE EXTERNAL SOURCE SCORES
    # Non-null scores must be between zero and one.
    # =====================================================================

    for c in ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']:
        df = check(df, f'INVALID_SCORE_{c}', (F.col(c) < 0) | (F.col(c) > 1), total, 'Score 0-1')

    # =====================================================================
    # SECTION 27: DEFINE CATEGORY DOMAINS
    # Set allowed values for contract, gender, and ownership fields.
    # =====================================================================

    domains = {'NAME_CONTRACT_TYPE':['Cash loans','Revolving loans'], 'CODE_GENDER':['M','F'],
               'FLAG_OWN_CAR':['Y','N'], 'FLAG_OWN_REALTY':['Y','N']}

    # =====================================================================
    # SECTION 28: VALIDATE CATEGORY DOMAINS
    # Flag category values outside the configured domains.
    # =====================================================================

    for c, values in domains.items():
        df = check(df, f'INVALID_CATEGORY_{c}', F.col(c).isNotNull() & ~F.col(c).isin(values), total, str(values))

    # =====================================================================
    # SECTION 29: CALCULATE RECORD HASH
    # Build a content hash used for deterministic duplicate ordering.
    # =====================================================================

    columns = [c for c in df.columns if c != '_reasons']
    df = df.withColumn('_hash', F.sha2(F.to_json(F.struct(*[F.col(c) for c in columns]),
                                               {'ignoreNullFields':'false'}), 256))

    # =====================================================================
    # SECTION 30: RANK DUPLICATE PRIMARY KEYS
    # Prefer newer Bronze ingestion timestamps when available.
    # =====================================================================

    order = ([F.col('BRONZE_INGESTION_TIMESTAMP').desc_nulls_last()]
             if 'BRONZE_INGESTION_TIMESTAMP' in df.columns else []) + [F.col('_hash').asc()]
    df = df.withColumn('_rank', F.row_number().over(Window.partitionBy('SK_ID_CURR').orderBy(*order)))

    # =====================================================================
    # SECTION 31: FLAG DUPLICATE PRIMARY KEYS
    # Retain one record per key and quarantine additional copies.
    # =====================================================================

    df = check(df, 'DUPLICATE_PRIMARY_KEY', F.col('SK_ID_CURR').isNotNull() & (F.col('_rank') > 1), total, 'Unique SK_ID_CURR')

    # =====================================================================
    # SECTION 32: CACHE AND SPLIT VALID / REJECTED ROWS
    # Avoid recomputation during quality counts and feature generation.
    # =====================================================================

    df = df.cache()
    try:
        good = df.filter(F.size('_reasons') == 0)
        bad = df.filter(F.size('_reasons') > 0)
        good_count, bad_count = good.count(), bad.count()
        if good_count + bad_count != total:
            raise RuntimeError('Row reconciliation failed')
        audit('CLASSIFICATION', total, good_count, bad_count)

        # =====================================================================
        # SECTION 33: BUILD QUARANTINE DATASET
        # Preserve rejected rows with rejection reasons and run metadata.
        # =====================================================================

        quarantine = (bad.withColumn('REJECTION_REASON', F.concat_ws('; ', '_reasons'))
                      .withColumn('SILVER_RUN_ID', F.lit(RUN_ID))
                      .withColumn('REJECTED_TIMESTAMP', F.lit(NOW).cast('timestamp'))
                      .drop('_reasons','_hash','_rank'))

        # =====================================================================
        # SECTION 34: BUILD CLEAN SILVER DATASET
        # Remove internal validation helper columns from accepted rows.
        # =====================================================================

        silver = good.drop('_reasons','_hash','_rank')

        # =====================================================================
        # SECTION 35: ADD NULL INDICATOR FEATURES
        # Create missingness flags without changing original values.
        # =====================================================================

        for c in ['AMT_ANNUITY','AMT_GOODS_PRICE','EXT_SOURCE_1','EXT_SOURCE_2',
                  'EXT_SOURCE_3','OCCUPATION_TYPE','NAME_TYPE_SUITE','CNT_FAM_MEMBERS']:
            if c in silver.columns:
                silver = silver.withColumn(f'{c}_IS_NULL', F.when(F.col(c).isNull(),1).otherwise(0))

        # =====================================================================
        # SECTION 35: ADD NULL INDICATOR FEATURES
        # Create missingness flags without changing original values.
        # =====================================================================

        for c in ['AMT_ANNUITY','AMT_GOODS_PRICE','EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']:
            q = silver.filter(F.col(c).isNotNull()).approxQuantile(c, [0.5], 0.01)
            silver = silver.withColumn(f'{c}_IMPUTED', F.coalesce(F.col(c), F.lit(q[0] if q else None).cast('double')))

        # =====================================================================
        # SECTION 37: CREATE AGE GROUPS
        # Categorize applicant age for reporting and modeling.
        # =====================================================================

        silver = silver.withColumn('AGE_GROUP', F.when(F.col('AGE_YEARS') < 25,'18-24')
                .when(F.col('AGE_YEARS') < 35,'25-34').when(F.col('AGE_YEARS') < 45,'35-44')
                .when(F.col('AGE_YEARS') < 55,'45-54').when(F.col('AGE_YEARS') < 65,'55-64').otherwise('65+'))

        # =====================================================================
        # SECTION 38: CREATE EMPLOYMENT CATEGORIES
        # Bucket years of employment and preserve unknown cases.
        # =====================================================================

        silver = silver.withColumn('EMPLOYMENT_CATEGORY', F.when(F.col('EMPLOYMENT_YEARS').isNull(),'Unknown')
                .when(F.col('EMPLOYMENT_YEARS') < 1,'<1 year').when(F.col('EMPLOYMENT_YEARS') < 3,'1-3 years')
                .when(F.col('EMPLOYMENT_YEARS') < 5,'3-5 years').when(F.col('EMPLOYMENT_YEARS') < 10,'5-10 years')
                .otherwise('10+ years'))

        # =====================================================================
        # SECTION 39: CREATE FINANCIAL RATIOS
        # Compute guarded ratios only when denominators are positive.
        # =====================================================================

        for name, (num, den) in {'CREDIT_INCOME_RATIO':('AMT_CREDIT','AMT_INCOME_TOTAL'),
                                'ANNUITY_INCOME_RATIO':('AMT_ANNUITY','AMT_INCOME_TOTAL'),
                                'CREDIT_ANNUITY_RATIO':('AMT_CREDIT','AMT_ANNUITY'),
                                'GOODS_CREDIT_RATIO':('AMT_GOODS_PRICE','AMT_CREDIT')}.items():
            silver = silver.withColumn(name, F.when(F.col(den) > 0, F.round(F.col(num)/F.col(den),4)))

        # =====================================================================
        # SECTION 40: CREATE INCOME BANDS
        # Use example analytical bands, not official risk thresholds.
        # =====================================================================

        silver = silver.withColumn('INCOME_CATEGORY', F.when(F.col('AMT_INCOME_TOTAL') < 100000,'LOW')
                .when(F.col('AMT_INCOME_TOTAL') < 250000,'MEDIUM')
                .when(F.col('AMT_INCOME_TOTAL') < 500000,'HIGH').otherwise('VERY_HIGH'))

        # =====================================================================
        # SECTION 41: CREATE CREDIT BANDS
        # Group credit amounts into example reporting ranges.
        # =====================================================================

        silver = silver.withColumn('CREDIT_CATEGORY', F.when(F.col('AMT_CREDIT') < 100000,'SMALL')
                .when(F.col('AMT_CREDIT') < 500000,'MEDIUM')
                .when(F.col('AMT_CREDIT') < 1000000,'LARGE').otherwise('VERY_LARGE'))

        # =====================================================================
        # SECTION 42: CALCULATE EXTERNAL SCORE AVERAGE
        # Average available external scores while ignoring null values.
        # =====================================================================

        score_cols = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
        score_sum = sum((F.coalesce(F.col(c),F.lit(0.0)) for c in score_cols), F.lit(0.0))
        score_count = sum((F.when(F.col(c).isNotNull(),1).otherwise(0) for c in score_cols), F.lit(0))
        silver = silver.withColumn('EXT_SOURCE_AVAILABLE_COUNT',score_count)
        silver = silver.withColumn('EXT_SOURCE_AVG',F.when(score_count > 0,F.round(score_sum/score_count,6)))

        # =====================================================================
        # SECTION 43: CREATE FAMILY FEATURES
        # Derive family size category and income per family member.
        # =====================================================================

        if 'CNT_FAM_MEMBERS' in silver.columns:
            silver = silver.withColumn('FAMILY_SIZE_CATEGORY',F.when(F.col('CNT_FAM_MEMBERS').isNull(),'UNKNOWN')
                    .when(F.col('CNT_FAM_MEMBERS') <= 2,'SMALL')
                    .when(F.col('CNT_FAM_MEMBERS') <= 4,'MEDIUM').otherwise('LARGE'))
            silver = silver.withColumn('INCOME_PER_FAMILY_MEMBER',F.when(F.col('CNT_FAM_MEMBERS') > 0,
                    F.round(F.col('AMT_INCOME_TOTAL')/F.col('CNT_FAM_MEMBERS'),2)))

        # =====================================================================
        # SECTION 44: ADD SILVER PROCESSING METADATA
        # Attach run identifier, processing time, source, and quality status.
        # =====================================================================

        silver = (silver.withColumn('SILVER_RUN_ID',F.lit(RUN_ID))
                  .withColumn('SILVER_PROCESSED_TIMESTAMP',F.lit(NOW).cast('timestamp'))
                  .withColumn('SILVER_SOURCE',F.lit('HOME_CREDIT_APPLICATION_TRAIN'))
                  .withColumn('SILVER_QUALITY_STATUS',F.lit('VALID')))
        # Full snapshot: overwrite existing Silver table/schema.

        # =====================================================================
        # SECTION 45: WRITE SILVER UNITY CATALOG TABLE
        # Overwrite the current Silver snapshot and update its schema.
        # =====================================================================

        silver.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{BASE}.`home_credit_silver`')
        audit('SILVER_WRITE',good_count,good_count)
        # Quarantine is overwritten per snapshot to avoid accumulating duplicate rejects.

        # =====================================================================
        # SECTION 46: WRITE QUARANTINE UNITY CATALOG TABLE
        # Overwrite the quarantine snapshot for the current Bronze input.
        # =====================================================================

        quarantine.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{BASE}.`home_credit_quarantine`')
        audit('QUARANTINE_WRITE',bad_count,bad_count)

        # =====================================================================
        # SECTION 47: WRITE VALIDATION LOG TABLE
        # Append validation outcomes to a Delta audit trail.
        # =====================================================================

        validation_schema = 'run_id string, validation_name string, status string, total_records long, failed_records long, description string, validation_timestamp timestamp'
        spark.createDataFrame(checks,validation_schema).write.format('delta').mode('append').saveAsTable(f'{BASE}.`home_credit_validation_logs`')
        audit('VALIDATION_LOG_WRITE',len(checks),len(checks))

        # =====================================================================
        # SECTION 48: WRITE AUDIT LOG TABLE
        # Append stage-level row counts and timestamps.
        # =====================================================================

        audit('AUDIT_LOG_WRITE',len(audits)+1,len(audits)+1)
        audit_schema = 'run_id string, stage string, input_records long, output_records long, rejected_records long, audit_timestamp timestamp'
        spark.createDataFrame(audits,audit_schema).write.format('delta').mode('append').saveAsTable(f'{BASE}.`home_credit_audit_logs`')

        # =====================================================================
        # SECTION 49: PRINT PIPELINE RECONCILIATION
        # Show counts and inspect selected Silver features.
        # =====================================================================

        print(f'COMPLETED run={RUN_ID} bronze={total} silver={good_count} quarantine={bad_count}')
        for name in ['home_credit_silver','home_credit_quarantine','home_credit_validation_logs','home_credit_audit_logs']:
            print(name, spark.table(f'{BASE}.`{name}`').count())
        spark.table(f'{BASE}.`home_credit_silver`').select('SK_ID_CURR','TARGET','AGE_YEARS','CREDIT_INCOME_RATIO').show(10,False)

    # =====================================================================
    # SECTION 50: RELEASE CACHED DATA
    # Unpersist the intermediate dataset even if a write fails.
    # =====================================================================

    finally:
        df.unpersist()



# =====================================================================
# SECTION 51: SCRIPT ENTRY POINT
# Run the pipeline when executed as a Python file.
# =====================================================================

if __name__ == '__main__':
    main()


---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-6053358296678864>, line 501
    495 # =====================================================================
    496 # SECTION 51: SCRIPT ENTRY POINT
    497 # Run the pipeline when executed as a Python file.
    498 # =====================================================================
    500 if __name__ == '__main__':
--> 501     main()

File <command-6053358296678864>, line 82, in main()
     80 spark.conf.set('spark.sql.session.timeZone', 'UTC')
     81 from pyspark.dbutils import DBUtils
---> 82 key = DBUtils(spark).secrets.get(scope='azure-storage', key='insurance-account-key')
     83 spark.conf.set(f'fs.azure.account.key.{ACCOUNT}.blob.core.windows.net', key)
     84 spark.sql(f'USE CATALOG `{CATALOG}`')

File /databricks/python_shell/lib/dbruntime/dbutils.py:449, in DBUtils.SecretsHandler.get(self, scope, catalog